Objective of this notebook is to implement MVN with Isotropic Covariance and MVN with AR(1) Covariance with different number of dimensions, and assess the performance of $\hat R_{ν}$

In [ ]:
!pip uninstall -yq jax jaxlib jax-cuda12-plugin jax-cuda12-pjrt tensorflow-probability
# heads out since jax might drop support for cuda 12, currently (as of Aug 10, 2026), JAX has issues with CUDA 13
# see this post: https://github.com/jax-ml/jax/issues/37923
!pip install -Uq "jax[cuda12]" tfp-nightly blackjax inference_gym optax

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 100.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 120.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 390.9/390.9 kB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 131.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 113.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.9/184.9 MB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 MB 30.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires tensorflow-probability>=0.13.0, which is not installed.


In [ ]:
# run those checks if package compatibility is in trouble

# import jax
# import tensorflow_probability as tfp
# import jaxlib

# print("jaxlib:", jaxlib.__version__)
# print("TFP:", tfp.__version__)

# !pip show jax
# !pip show jaxlib
# !pip show blackjax

# import jax.numpy as jnp

# import blackjax

# import tensorflow_probability.substrates.jax as tfp
# import inference_gym.using_jax as gym

# print("JAX:", jax.__version__)
# print("BlackJAX:", blackjax.__version__)
# print("TFP:", tfp.__version__)
# print("ArviZ:", avs.__version__)
# print("Inference Gym imported successfully!")

**Package Import and Other Setups**

In [ ]:
# GPU set up to accelerate performance
import os
# in case jax eats up my GPU RAM
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ['XLA_FLAGS'] = (
    '--xla_gpu_triton_gemm_any=True '
    '--xla_gpu_enable_latency_hiding_scheduler=true '
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp
from jax import random, jit, vmap, lax
import tensorflow_probability.substrates.jax as tfp
tfd = tfp.distributions
import inference_gym.using_jax as gym
import jaxlib
import blackjax

import optax

# import arviz as az
# import arviz_stats as avs

import warnings
warnings.filterwarnings('ignore')

import psutil

import gc

from google.colab import drive
from matplotlib.lines import Line2D

process = psutil.Process(os.getpid())

def mem(msg):
    print(f"{msg}: {process.memory_info().rss / 1024**2:.1f} MB")

# verification to make sure this is on a GPU
print(jax.devices())
print(jax.default_backend())

[CudaDevice(id=0)]
gpu


In [ ]:
drive.mount('/content/drive')
utility_link = '/content/drive/MyDrive/JHU Stuff/Capstone/Utility_Functions/Dimension_Utility.py'
with open(utility_link) as f: exec(f.read())

Mounted at /content/drive


**Hyperparameter Setups**

In [ ]:
max_warmup = 1000
warmup_window = 100

window_array = np.append(np.repeat(10, 10),
                      np.repeat(warmup_window, max_warmup // warmup_window - 1))

warmup_length = np.repeat(10, len(window_array))
for i in range(len(warmup_length) - 1):
    warmup_length[i + 1] = warmup_length[i] + window_array[i + 1]

# Transition kernel for short regime
repitition = 10
num_chains_short = 2048
num_super_chains = 16

In [ ]:
# quantiles for chi squared with df = 1
chi_up = 3.841459 # 95th quantile for chi squared with df = 1
chi_lo = 0.00393214  # 05th quantile for chi squared with df = 1
tau = 1e-4
M = num_chains_short // num_super_chains
nRhat_lower = np.sqrt(1 + 1 / M + tau)
eps_lower = nRhat_lower - 1
bound = [chi_lo / num_chains_short, chi_up / num_chains_short]
threshold = eps_lower

**MVN with Isotropic Covariance**

In [ ]:
#simulation part:
Iso_MSE_c_list = []
Iso_MSE_n_list = []
Iso_RHat_c_list = []
Iso_RHat_n_list = []

In [ ]:
dimension_list = [10,20,50,100]
run_simulation(Iso_builder, dimension_list, warmup_length, repitition,
               num_chains_short,num_super_chains,
               Iso_RHat_c_list,Iso_MSE_c_list,
               Iso_RHat_n_list,Iso_MSE_n_list)

Simulation Start, D=10: 1598.8 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.00044417165918275714
Naive initialization. Warmup Length: 10; mean of MSE is: 0.0004462952201720327
Simulation Start, D=10: 1855.3 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.00042357048369012773
Naive initialization. Warmup Length: 20; mean of MSE is: 0.00044522417010739446
Simulation Start, D=10: 1861.4 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.00038096492062322795
Naive initialization. Warmup Length: 30; mean of MSE is: 0.00041149748722091317
Simulation Start, D=10: 1862.4 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.00048509068437851965
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0005364917451515794
Simulation Start, D=10: 1864.4 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.000390913657611236
Naive initialization. Warmup Length: 50; mean of MSE is: 0.00033089888165704906
Simu

In [ ]:
MSE_c_df = pd.DataFrame(Iso_MSE_c_list)
R_Hat_c_df = pd.DataFrame(Iso_RHat_c_list)

MSE_n_df = pd.DataFrame(Iso_MSE_n_list)
R_Hat_n_df = pd.DataFrame(Iso_RHat_n_list)

In [ ]:
MSE_c_df.to_pickle(
    '/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/DimensionExperiment/Iso_MSE_c.pkl'
)

MSE_n_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/DimensionExperiment/Iso_MSE_n.pkl"
)

R_Hat_c_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/DimensionExperiment/Iso_Rhat_c.pkl"
)

R_Hat_n_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/DimensionExperiment/Iso_Rhat_n.pkl"
)

**MVN With AR(1) Covariance**

In [ ]:
#simulation part:
AR_MSE_c_list = []
AR_MSE_n_list = []
AR_RHat_c_list = []
AR_RHat_n_list = []

In [ ]:
dimension_list = [10,20,50,100]
run_simulation(ar_builder, dimension_list, warmup_length, repitition,
               num_chains_short,num_super_chains,
               AR_RHat_c_list,AR_MSE_c_list,
               AR_RHat_n_list,AR_MSE_n_list)

Simulation Start, D=10: 1455.3 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.0005572371301241219
Naive initialization. Warmup Length: 10; mean of MSE is: 0.00047761821770109236
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.0005818491918034852
Naive initialization. Warmup Length: 20; mean of MSE is: 0.0005695262807421386
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.0005274390568956733
Naive initialization. Warmup Length: 30; mean of MSE is: 0.0005021203542128205
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0005248984671197832
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0005036401562392712
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.00047888405970297754
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0005681755137629807
Constrained initialization. Warmup Length: 60; mean of MSE is: 0.0004195308720227331
Naive initialization. Warmup Length: 60; mean of MSE is: 0.

In [ ]:
MSE_c_df = pd.DataFrame(AR_MSE_c_list)
R_Hat_c_df = pd.DataFrame(AR_RHat_c_list)

MSE_n_df = pd.DataFrame(AR_MSE_n_list)
R_Hat_n_df = pd.DataFrame(AR_RHat_n_list)

In [ ]:
MSE_c_df.to_pickle(
    '/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/DimensionExperiment/AR_MSE_c.pkl'
)

MSE_n_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/DimensionExperiment/AR_MSE_n.pkl"
)

R_Hat_c_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/DimensionExperiment/AR_Rhat_c.pkl"
)

R_Hat_n_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/DimensionExperiment/AR_Rhat_n.pkl"
)

In [ ]:
# # auto disconnect
# from google.colab import runtime
# runtime.unassign()